# Malay-Specific BERT — Zero-Shot and Fine-Tuned
Third transformer comparison: does a Malay-specific model outperform generic multilingual XLM-R on code-switched Manglish text?

In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.metrics import classification_report, accuracy_score, cohen_kappa_score, confusion_matrix
import os
import warnings
warnings.filterwarnings('ignore')

device = 0 if torch.cuda.is_available() else -1
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Device: {"GPU" if device == 0 else "CPU"}')

CUDA available: True
Device: GPU


## 1. Load Gold Test Set

In [2]:
test_df = pd.read_csv('../data/NLP_Dataset_preparation/gold_labeled.csv')
y_test = test_df['human_label']
print(f'Test rows: {len(test_df)}')
print(test_df['human_label'].value_counts())

Test rows: 402
human_label
neutral     241
negative    124
positive     37
Name: count, dtype: int64


## 2. Check Model Availability
mesolitica/malaysian-mistral-7b is too large for this task; testing smaller
Malay-specific sentiment models from mesolitica/malaya-ai on Hugging Face Hub.

## 3. Zero-Shot Test on Best Available Malay Model
Manually set MODEL_NAME below once Cell 6 confirms which model is actually available.

In [3]:
MODEL_NAME = "mesolitica/bert-base-standard-bahasa-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Loaded tokenizer for {MODEL_NAME}')

Loaded tokenizer for mesolitica/bert-base-standard-bahasa-cased


## 4. Load and Preprocess Training Data

In [4]:
import re

train_df = pd.read_csv('../data/malaysian_sentiment_labeled.csv')

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'[*_]{1,2}([^*_]+)[*_]{1,2}', r'\1', text)
    text = re.sub(r'^>.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'&gt;|&amp;|&lt;|&nbsp;', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['cleaned'] = train_df['body'].apply(preprocess)
test_df['cleaned'] = test_df['body'].apply(preprocess)
train_df = train_df[train_df['cleaned'].str.split().apply(len) >= 3].reset_index(drop=True)

label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label = {v: k for k, v in label2id.items()}
train_df['label'] = train_df['sentiment'].map(label2id)
test_df['label'] = test_df['human_label'].map(label2id)
print(f'Train: {len(train_df)}  Test: {len(test_df)}')

Train: 7174  Test: 402


## 5. Fine-Tune (settings matching the winning XLM-R run: 6 epochs, capped weights)

In [ ]:
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments
from torch.nn import CrossEntropyLoss

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding='max_length',
                              max_length=self.max_len, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = SentimentDataset(train_df['cleaned'].tolist(), train_df['label'].tolist(), tokenizer)
test_dataset = SentimentDataset(test_df['cleaned'].tolist(), test_df['label'].tolist(), tokenizer)

import transformers.modeling_utils as mu
mu.check_torch_load_is_safe = lambda: None

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
).to('cuda' if torch.cuda.is_available() else 'cpu')

class_counts = train_df['label'].value_counts().sort_index()
raw_weights = [len(train_df) / (3 * c) for c in class_counts]
class_weights = torch.tensor([min(w, 5.0) for w in raw_weights], dtype=torch.float).to(model.device)
print(f'Class weights: {class_weights}')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir='../models/transformers/malay_bert_checkpoints',
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    logging_steps=50,
    save_strategy='no',
    eval_strategy='no',
    fp16=torch.cuda.is_available(),
    report_to='none'
)

trainer = WeightedTrainer(model=model, args=training_args, train_dataset=train_dataset)
trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: mesolitica/bert-base-standard-bahasa-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISS

Class weights: tensor([1.1969, 0.4886, 5.0000], device='cuda:0')


Step,Training Loss
50,2.095114
100,2.104144
150,1.908323
200,2.032182
250,1.790759


## 6. Evaluate

In [ ]:
import numpy as np
preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=1)
pred_labels = [id2label[p] for p in preds]
y_test_list = test_df['human_label'].tolist()

acc = accuracy_score(y_test_list, pred_labels)
kappa = cohen_kappa_score(y_test_list, pred_labels)
report = classification_report(y_test_list, pred_labels, target_names=['negative','neutral','positive'], output_dict=True)

print(f'Malay BERT Fine-Tuned (6 epochs)')
print(f'Accuracy: {acc*100:.1f}%   Kappa: {kappa:.3f}')
print(classification_report(y_test_list, pred_labels, target_names=['negative','neutral','positive']))

labels = ['negative','neutral','positive']
cm = confusion_matrix(y_test_list, pred_labels, labels=labels)
print(f"{'':>10}" + "".join(f"{l:>10}" for l in labels))
for i, l in enumerate(labels):
    print(f"{l:>10}" + "".join(f"{cm[i][j]:>10}" for j in range(len(labels))))

## 7. Save Model and Results

In [ ]:
SAVE_DIR = '../models/transformers/malay_bert_finetuned'
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f'Saved to {SAVE_DIR}')

RESULTS_FILE = '../results/model_results.csv'
new_row = pd.DataFrame([{
    'model': 'Malay BERT Fine-Tuned (6 epochs)',
    'status': 'transformer_finetuned',
    'accuracy': acc,
    'kappa': kappa,
    'neg_f1': report['negative']['f1-score'],
    'neu_f1': report['neutral']['f1-score'],
    'pos_f1': report['positive']['f1-score'],
}])
existing = pd.read_csv(RESULTS_FILE)
existing = existing[existing['model'] != 'Malay BERT Fine-Tuned (6 epochs)']
combined = pd.concat([existing, new_row], ignore_index=True)
combined.to_csv(RESULTS_FILE, index=False)
print(combined.to_string(index=False))